# Did Colfax construction reduce serious injuries and deaths?

Severe crashes on Colfax Ave between Broadway and Colorado Blvd versus the rest of Denver, Jan 1 - Sep 27 of each year. Pre period 2013-2024; construction period 2025-2026. The analysis logic lives in `analysis/colfax_analysis.py`; interpretation is in `report/colfax_report.qmd`.

Two definitions of a "severe" crash are shown side by side throughout, because they behave differently (see the data-quality cell):

- **A: offense code.** Top offense is `TRAF - ACCIDENT - FATAL` or `TRAF - ACCIDENT - SBI` (reproduces the 318 SBI events quoted for 2025).
- **B: count columns.** `SERIOUSLY_INJURED` or `FATALITIES` is greater than zero.

Reading the tables: `colfax_post` below 1 means Colfax fell more than the citywide baseline; `alt_post` above 1 means 13th/14th Ave fell less than the baseline.

In [ ]:
import importlib
import sys
from pathlib import Path

import polars as pl

root = Path.cwd() if (Path.cwd() / "analysis").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "analysis"))
import colfax_analysis as ca

importlib.reload(ca)  # pick up edits to the module without restarting the kernel

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
df, geom = ca.prepare()
df_all = df.with_columns(pl.lit(True).alias("all_crashes"))
R = {
    o: ca.run_primary(
        df_all if o == "all_crashes" else df,
        geom,
        n_sims=199 if o == "all_crashes" else 999,
        outcome=o,
    )
    for o in ("severe", "severe_counts", "fatal_offense", "all_crashes")
}
A, B = R["severe"], R["severe_counts"]
LAB_A, LAB_B = "A: offense code", "B: count columns"


def side(key, cols=None):
    """Stack one result table for definition A (offense code) and B (count columns)."""
    out = pl.concat(
        [
            A[key].with_columns(pl.lit(LAB_A).alias("definition")),
            B[key].with_columns(pl.lit(LAB_B).alias("definition")),
        ]
    )
    out = out.select(["definition"] + [c for c in out.columns if c != "definition"])
    return out.select(cols) if cols else out


ca.reconcile_2025(df)

In [ ]:
# Data quality: offense code vs count columns (all of Denver, by year)
display(ca.offense_vs_counts(df))
# Jan 1 - Sep 27 severe-crash counts under both definitions
ca.definitions_by_year(df)

In [ ]:
# Counts by zone, both definitions
pl.concat(
    [
        A["counts"].pivot(on="group", index="year", values="n").with_columns(pl.lit("A").alias("def")),
        B["counts"].pivot(on="group", index="year", values="n").with_columns(pl.lit("B").alias("def")),
    ]
).select("def", "year", "colfax", "alt", "rest").sort("def", "year")

In [ ]:
# Primary test: Poisson difference-in-differences, both definitions
cols = ["definition", "term", "rate_ratio", "ci_low", "ci_high", "p_value"]
display(side("did", cols))
display(side("did_by_year", cols))
display(side("did_trend", cols))
display(side("pretrend", cols))
pl.concat(
    [
        pl.DataFrame([r["fisher_colfax"], r["fisher_alt"]]).with_columns(pl.lit(lab).alias("definition"))
        for r, lab in ((A, LAB_A), (B, LAB_B))
    ]
).select("definition", "group", "pre_group", "post_group", "pre_rest", "post_rest", "odds_ratio", "ci_low", "ci_high", "p_value")

In [ ]:
# Placebo corridors, both definitions
display(side("placebo_sens", ["definition", "baseline_ratio", "observed_ratio", "n_windows", "p_all_windows", "n_independent", "p_independent", "percentile"]))
display(ca.fig_placebo(A["placebo"], " (A: offense code)"))
ca.fig_placebo(B["placebo"], " (B: count columns)")

In [ ]:
# Spatial scan, both definitions (map shows definition A)
rows = []
for r, lab in ((A, LAB_A), (B, LAB_B)):
    S = r["scan"]
    rows += [
        {"definition": lab, "test": "citywide (most extreme cold cluster)", "crashes_in_circle": S["best_n"], "construction_observed": S["best_c"], "construction_expected": S["best_expected"], "llr": S["best_llr"], "p_value": S["p_global"]},
        {"definition": lab, "test": "focused on Colfax", "crashes_in_circle": S["focus_best_n"], "construction_observed": S["focus_best_c"], "construction_expected": S["focus_best_expected"], "llr": S["focus_best_llr"], "p_value": S["p_focus"]},
    ]
display(pl.DataFrame(rows))
ca.fig_scan(df, geom, A["scan"])

In [ ]:
# Trend plots (A then B) and corridor map (A)
display(ca.fig_trend(A["counts"], " (A: offense code)"))
display(ca.fig_trend(B["counts"], " (B: count columns)"))
ca.fig_corridor_map(df, geom)

In [ ]:
# Sensitivity (Colfax and alt-route terms), buffer widths, and other outcomes
def sens_rows(r, label):
    parts = [
        r["did"].with_columns(pl.lit("primary").alias("model")),
        r["did_trend"].filter(pl.col("term").is_in(["colfax_post", "alt_post"])).with_columns(pl.lit("zone-specific linear trends").alias("model")),
        r["did_no2020"].with_columns(pl.lit("excluding 2020").alias("model")),
        r["did_2021_on"].with_columns(pl.lit("pre period 2021-2024 only").alias("model")),
    ]
    return pl.concat(parts).with_columns(pl.lit(label).alias("definition"))


display(pl.concat([sens_rows(A, LAB_A), sens_rows(B, LAB_B)]).select("definition", "model", "term", "rate_ratio", "ci_low", "ci_high", "p_value"))
display(
    pl.concat(
        [
            ca.buffer_sensitivity(df, geom, outcome="severe").with_columns(pl.lit(LAB_A).alias("definition")),
            ca.buffer_sensitivity(df, geom, outcome="severe_counts").with_columns(pl.lit(LAB_B).alias("definition")),
        ]
    ).select("definition", "buffer_ft", "n_colfax", "term", "rate_ratio", "ci_low", "ci_high", "p_value")
)
pl.concat(
    [R[o]["did"].with_columns(pl.lit(o).alias("outcome")) for o in ("fatal_offense", "all_crashes")]
).select("outcome", "term", "rate_ratio", "ci_low", "ci_high", "p_value")

In [ ]:
# Small-count checks: negative binomial (alpha ~ 0 means no over-dispersion) and exact conditional Poisson
display(
    pl.concat(
        [
            A["did_nb"].with_columns(pl.lit(LAB_A).alias("definition"), pl.lit(A["nb_alpha"]).alias("alpha")),
            B["did_nb"].with_columns(pl.lit(LAB_B).alias("definition"), pl.lit(B["nb_alpha"]).alias("alpha")),
        ]
    ).select("definition", "term", "rate_ratio", "ci_low", "ci_high", "p_value", "alpha")
)
pl.concat(
    [
        A["exact_conditional"].with_columns(pl.lit(LAB_A).alias("definition")),
        B["exact_conditional"].with_columns(pl.lit(LAB_B).alias("definition")),
    ]
).select(
    "definition", "group", "pre_period", "pre_count", "post_count", "citywide_rate_ratio",
    "expected_post_count", "relative_rate_ratio", "ci_low", "ci_high", "p_lower_tail", "p_two_sided",
)

In [ ]:
# Combined corridor: Colfax + 13th/14th Ave as one unit vs the rest of the city
CMB = {
    "A": ca.combined_analysis(df, "severe"),
    "B": ca.combined_analysis(df, "severe_counts"),
    "all": ca.combined_analysis(df_all, "all_crashes"),
}


def cmb_rows(key, label):
    r = CMB[key]
    parts = [
        r["did"].with_columns(pl.lit("primary").alias("model")),
        r["did_by_year"].with_columns(pl.lit("by year").alias("model")),
        r["did_trend"].filter(pl.col("term") == "corridor_post").with_columns(pl.lit("zone-specific linear trend").alias("model")),
        r["did_no2020"].with_columns(pl.lit("excluding 2020").alias("model")),
        r["did_2021_on"].with_columns(pl.lit("pre period 2021-2024 only").alias("model")),
    ]
    return pl.concat(parts).with_columns(pl.lit(label).alias("definition"))


display(
    pl.concat(
        [
            CMB["A"]["components"].with_columns(pl.lit(LAB_A).alias("definition")),
            CMB["B"]["components"].with_columns(pl.lit(LAB_B).alias("definition")),
        ]
    ).select("definition", "group", "construction", "n", "years", "per_year")
)
display(
    pl.concat([cmb_rows("A", LAB_A), cmb_rows("B", LAB_B), cmb_rows("all", "all crashes")]).select(
        "definition", "model", "term", "rate_ratio", "ci_low", "ci_high", "p_value"
    )
)
pl.concat(
    [
        CMB["A"]["exact"].with_columns(pl.lit(LAB_A).alias("definition")),
        CMB["B"]["exact"].with_columns(pl.lit(LAB_B).alias("definition")),
        CMB["all"]["exact"].with_columns(pl.lit("all crashes").alias("definition")),
    ]
).select(
    "definition", "pre_period", "pre_count", "post_count", "citywide_rate_ratio",
    "expected_post_count", "relative_rate_ratio", "ci_low", "ci_high", "p_lower_tail", "p_two_sided",
)

In [ ]:
# Leave-one-pre-period-year-out sensitivity (Colfax, 13th/14th Ave, combined corridor)
LOO = {o: ca.leave_one_year_out(df, o) for o in ("severe", "severe_counts")}
loo = pl.concat(
    [
        LOO["severe"].with_columns(pl.lit(LAB_A).alias("definition")),
        LOO["severe_counts"].with_columns(pl.lit(LAB_B).alias("definition")),
    ]
)
display(loo.with_columns(pl.col("rate_ratio").round(2)).pivot(on="unit", index=["definition", "omitted_year"], values="rate_ratio"))
loo.filter(pl.col("omitted_year") != "none (all years)").group_by("definition", "unit").agg(
    pl.col("rate_ratio").min().alias("rr_min"),
    pl.col("rate_ratio").max().alias("rr_max"),
    pl.col("p_value").min().alias("p_min"),
    pl.col("p_value").max().alias("p_max"),
).sort("definition", "unit")

In [ ]:
# Wider corridor: 13th, 14th, Colfax, 16th and 17th Ave as one unit vs the rest of the city
dfw, geomw = ca.prepare_wide()
dfw_all = dfw.with_columns(pl.lit(True).alias("all_crashes"))
WD = {
    "A": ca.wide_analysis(dfw, "severe"),
    "B": ca.wide_analysis(dfw, "severe_counts"),
    "all": ca.wide_analysis(dfw_all, "all_crashes"),
}


def wide_rows(key, label):
    r = WD[key]
    parts = [
        r["did"].with_columns(pl.lit("primary").alias("model")),
        r["did_by_year"].with_columns(pl.lit("by year").alias("model")),
        r["did_trend"].filter(pl.col("term") == "corridor_post").with_columns(pl.lit("zone-specific linear trend").alias("model")),
        r["did_no2020"].with_columns(pl.lit("excluding 2020").alias("model")),
        r["did_2021_on"].with_columns(pl.lit("pre period 2021-2024 only").alias("model")),
    ]
    return pl.concat(parts).with_columns(pl.lit(label).alias("definition"))


display(
    pl.concat(
        [
            WD["A"]["by_street"].with_columns(pl.lit(LAB_A).alias("definition")),
            WD["B"]["by_street"].with_columns(pl.lit(LAB_B).alias("definition")),
        ]
    ).select("definition", "zone", "construction", "n", "years", "per_year")
)
display(pl.concat([wide_rows("A", LAB_A), wide_rows("B", LAB_B), wide_rows("all", "all crashes")]).select("definition", "model", "term", "rate_ratio", "ci_low", "ci_high", "p_value"))
display(
    pl.concat(
        [
            WD["A"]["exact"].with_columns(pl.lit(LAB_A).alias("definition")),
            WD["B"]["exact"].with_columns(pl.lit(LAB_B).alias("definition")),
            WD["all"]["exact"].with_columns(pl.lit("all crashes").alias("definition")),
        ]
    ).select("definition", "pre_period", "pre_count", "post_count", "expected_post_count", "relative_rate_ratio", "ci_low", "ci_high", "p_two_sided")
)
# Colfax vs the four other streets pooled, each relative to the rest of the city
display(
    pl.concat(
        [
            WD["A"]["did_three_group"].with_columns(pl.lit(LAB_A).alias("definition")),
            WD["B"]["did_three_group"].with_columns(pl.lit(LAB_B).alias("definition")),
            WD["all"]["did_three_group"].with_columns(pl.lit("all crashes").alias("definition")),
        ]
    ).select("definition", "term", "rate_ratio", "ci_low", "ci_high", "p_value")
)
pl.concat(
    [
        ca.wide_buffer_sensitivity(dfw, geomw, outcome="severe").with_columns(pl.lit(LAB_A).alias("definition")),
        ca.wide_buffer_sensitivity(dfw, geomw, outcome="severe_counts").with_columns(pl.lit(LAB_B).alias("definition")),
    ]
).select("definition", "buffer_ft", "n_corridor", "rate_ratio", "ci_low", "ci_high", "p_value")

## Full-calendar version: seasonality handled in the model

Drops the Jan 1 - Sep 27 window and uses every month from Jan 2013 to Sep 27, 2026 (zone x month counts, with days observed as a Poisson offset). Seasonality is removed with month-of-year effects (primary), month-of-sample effects, or zone-specific harmonics; Oct-Dec 2024 now counts as construction. Placebo, scan and exact tests are not repeated here.

In [ ]:
# Full-calendar DiD: seasonality in the model instead of the Jan-Sep 27 window
FC = {o: ca.run_fullcalendar(df, o) for o in ("severe", "severe_counts")}
for o in FC:
    FC[o]["wide"] = ca.run_fullcalendar_wide(dfw, o)
display({k: v["profile_test"] for k, v in FC.items()})
display(ca.fig_seasonal_profile({LAB_A: FC["severe"]["profile"], LAB_B: FC["severe_counts"]["profile"]}))
display(
    pl.concat(
        [
            FC["severe"]["specs"].with_columns(pl.lit(LAB_A).alias("definition")),
            FC["severe_counts"]["specs"].with_columns(pl.lit(LAB_B).alias("definition")),
        ]
    ).select("definition", "spec", "term", "rate_ratio", "ci_low", "ci_high", "p_value")
)
display(
    pl.concat(
        [
            FC["severe"]["combined_specs"].with_columns(pl.lit("Colfax + 13th / 14th").alias("unit"), pl.lit(LAB_A).alias("definition")),
            FC["severe_counts"]["combined_specs"].with_columns(pl.lit("Colfax + 13th / 14th").alias("unit"), pl.lit(LAB_B).alias("definition")),
            FC["severe"]["wide"]["combined_specs"].with_columns(pl.lit("Wider corridor").alias("unit"), pl.lit(LAB_A).alias("definition")),
            FC["severe_counts"]["wide"]["combined_specs"].with_columns(pl.lit("Wider corridor").alias("unit"), pl.lit(LAB_B).alias("definition")),
        ]
    ).select("unit", "definition", "spec", "rate_ratio", "ci_low", "ci_high", "p_value")
)
ca.fig_fullcal_forest({LAB_A: FC["severe"]["specs"], LAB_B: FC["severe_counts"]["specs"]})